# Phishing URL Detection with NLP

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, Dropout
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import re

np.random.seed(42)
legit_domains = ['google.com', 'facebook.com', 'amazon.com', 'microsoft.com', 'apple.com']
phishing_patterns = ['secure-', 'verify-', 'update-', 'login-', 'account-', 'bank-']
suspicious_tlds = ['.tk', '.ml', '.ga', '.cf', '.pw']

urls, labels = [], []
for _ in range(2500):
    domain = np.random.choice(legit_domains)
    url = f"https://{domain}{np.random.choice(['', '/login', '/home', '/about'])}"
    urls.append(url)
    labels.append(0)

for _ in range(2500):
    pattern = np.random.choice(phishing_patterns)
    fake_domain = pattern + np.random.choice(legit_domains).replace('.', '-')
    if np.random.random() < 0.3:
        fake_domain = fake_domain.replace('.com', np.random.choice(suspicious_tlds))
    url = f"http://{fake_domain}{np.random.choice(['/login', '/verify', '/update'])}"
    urls.append(url)
    labels.append(1)

df = pd.DataFrame({'url': urls, 'label': labels})

def preprocess_url(url):
    url = url.lower()
    url = re.sub(r'https?://', '', url)
    url = re.sub(r'[^\w\-\.]', ' ', url)
    return url

processed_urls = df['url'].apply(preprocess_url)

tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,3))
X_tfidf = tfidf.fit_transform(processed_urls)
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, df['label'], test_size=0.2, random_state=42)

lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)
print("TF-IDF + Logistic Regression:")
print(f"Accuracy: {np.mean(y_pred == y_test):.4f}")
print(classification_report(y_test, y_pred, target_names=['Safe', 'Phishing']))

tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=3000)
tokenizer.fit_on_texts(processed_urls)
sequences = tokenizer.texts_to_sequences(processed_urls)
X_seq = tf.keras.preprocessing.sequence.pad_sequences(sequences, maxlen=50)

X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(X_seq, df['label'], test_size=0.2, random_state=42)

lstm_model = Sequential([
    Embedding(3000, 64, input_length=50),
    LSTM(32, dropout=0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.fit(X_train_seq, y_train_seq, epochs=10, batch_size=32, validation_split=0.2, verbose=0)

y_pred_lstm = (lstm_model.predict(X_test_seq) > 0.5).astype(int).flatten()
print("\nLSTM Model:")
print(f"Accuracy: {np.mean(y_pred_lstm == y_test_seq):.4f}")
print(classification_report(y_test_seq, y_pred_lstm, target_names=['Safe', 'Phishing']))

test_urls = ["https://google.com/login", "http://secure-paypal-verify.tk/login", "https://amazon.com/product"]
for url in test_urls:
    processed = preprocess_url(url)
    tfidf_pred = lr_model.predict_proba(tfidf.transform([processed]))[0][1]
    seq = tokenizer.texts_to_sequences([processed])
    padded = tf.keras.preprocessing.sequence.pad_sequences(seq, maxlen=50)
    lstm_pred = lstm_model.predict(padded)[0][0]
    final_pred = "PHISHING" if (tfidf_pred + lstm_pred) / 2 > 0.5 else "SAFE"
    print(f"\n{url}: {final_pred} (TF-IDF: {tfidf_pred:.3f}, LSTM: {lstm_pred:.3f})")